# Ñ contra 16 bibliotecas de otros lenguajes, en las dos T4 de Kaggle

Las mismas seis pruebas (`matmul` 1024 y 2048 —y `matmul32`, lo mismo en precisión simple, en Ñ y PyTorch—, `seno`, `x*2+1` y `suma` de 2^24 números, `softmax` 4096×1024 y `fft` de 2^20) con **los mismos datos** en Ñ (C1EÑ en jit, nativo y GPU) y en:

| lenguaje | biblioteca | aquí | por defecto |
|---|---|---|---|
| Python | PyTorch | CPU y GPU | **activo** |
| C++ | Eigen | CPU | **activo** |
| C | OpenBLAS | CPU (BLAS solo hace matmul; el resto, bucles C con OpenMP) | **activo** |
| JavaScript | TensorFlow.js (tfjs-node) | CPU, **float32** | **activo** |
| Go | Gorgonia | CPU | **activo** |
| Fortran | Intel MKL | CPU | **activo** |
| Julia | CUDA.jl | CPU y GPU | apagado (instalar ~10–15 min) |
| Rust | Candle | CPU y GPU | apagado (compilar con CUDA ~15–25 min) |
| R | torch | CPU y GPU | apagado (~2 GB) |
| Java | ND4J | CPU | apagado |
| MATLAB | **Octave en su lugar** (MATLAB necesita licencia) | CPU | apagado |
| C# | TorchSharp | CPU y GPU | apagado (~2,5 GB) |
| Swift | **Swift sin Metal** (Metal solo existe en Apple) | CPU, un hilo | apagado |
| Scala | Breeze | CPU | apagado |
| Kotlin | KotlinDL (se mide TensorFlow 1.15 para Java, su motor) | CPU | apagado |

### Dos cuadernos: uno con las T4, otro solo CPU

Este mismo cuaderno se importa **dos veces** en Kaggle y se corren a la vez:

| | `PAPEL = 'gpu'` | `PAPEL = 'cpu'` |
|---|---|---|
| Acelerador | **GPU T4 x2** | **ninguno** (sesión de CPU) |
| Qué mide | las pruebas de **GPU**: PyTorch, CUDA.jl, Candle, R torch, TorchSharp y Ñ, dos a la vez (una por T4) | todas las pruebas de **CPU** de todos los lenguajes (también la parte CPU de PyTorch, Julia, Candle, R y TorchSharp), una tras otra, **con la CPU entera** |
| Qué instala | solo los lenguajes que tienen GPU | lo de CPU de todos |

Así cada ranking sale de **una sola máquina** (todas las CPU en la de CPU, todas las GPU en la de GPU), la CPU no se reparte, la instalación va en dos máquinas a la vez y las horas de GPU solo se gastan en GPU. Cada cuaderno guarda `comparar-lenguajes-gpu.csv` o `comparar-lenguajes-cpu.csv`; para la tabla conjunta, en uno de los dos añade como entrada (*Add Input → Your Work*) la salida del otro y vuelve a correr la última parte: la celda de juntar lo encuentra sola. Con `PAPEL = 'todo'` hace las dos cosas en una sola sesión de GPU.

### Cómo se reparte el trabajo dentro de cada cuaderno

1. **Instalar y compilar, varios a la vez.** Es lo que más tarda (bajar Julia, compilar Rust, libtorch de R y de .NET…) y casi todo es esperar a la red o al compilador, así que se hacen `INSTALAR_A_LA_VEZ` (3) lenguajes en paralelo. `apt-get` va de uno en uno porque no admite dos a la vez.
2. **Medir.** En el de GPU, dos trabajadores sacan lenguajes de una cola, uno por T4 (cada programa solo ve su GPU con `CUDA_VISIBLE_DEVICES` y queda atado a su mitad de núcleos con `taskset`), de los más largos a los más cortos. En el de CPU, uno detrás de otro con la CPU entera, para que ninguno le quite núcleos a otro.

Con `PAPEL = 'todo'` (una sola sesión de GPU) se puede elegir `MODO = 'paralelo'` (dos a la vez, cada uno con media CPU: los tiempos de CPU salen con media máquina) o `'secuencial'`.

Cada uno escribe `RES,prueba,tamaño,dispositivo,ms,huella,precisión`. La **huella** (suma del resultado) se compara con la de NumPy en doble precisión. Si un lenguaje falla al instalarse o al correr, se salta, se apunta por qué y el resto sigue.

**Por qué algunos solo en CPU:** Eigen, OpenBLAS, MKL, Gorgonia y Breeze son de CPU. TensorFlow.js para GPU, ND4J para CUDA y TensorFlow 1.15 (KotlinDL) piden CUDA 10/11, y Kaggle tiene CUDA 12. MATLAB y Metal no se pueden instalar aquí: sus sustitutos están marcados como tales.

**El Output queda liviano:** el motor de Ñ, Node y los lenguajes se instalan en `/kaggle/temp`, que Kaggle no guarda; en el Output (`/kaggle/working`) solo quedan los CSV y el JSON (unos KB). Lo que versiones viejas dejaban allí se borra al empezar.

**Antes de empezar:** en el cuaderno de GPU, *Settings → Accelerator → GPU T4 x2*; en el de CPU, *Accelerator → None*. En los dos, *Internet → On*.

In [ ]:
# 1 · Qué corre y cómo. Cambia y vuelve a correr desde aquí.
PAPEL = 'gpu'                # 'gpu' (sesión T4 x2: solo las pruebas de GPU) · 'cpu' (sesión sin acelerador: todas las de CPU) · 'todo'
ESCALA = 'normal'            # 'chica' para probar que todo corre en pocos minutos
MODO = 'paralelo'            # solo con PAPEL = 'todo': 'paralelo' (dos a la vez, uno por GPU) · 'secuencial'
INSTALAR_A_LA_VEZ = 3        # cuántos lenguajes se instalan/compilan a la vez
TODOS = False                # True: enciende los 16 sin tocar la lista de abajo
ACTIVOS = {
    'Ñ': True,
    'PyTorch': True, 'Eigen': True, 'OpenBLAS': True, 'TensorFlow.js': True, 'Gorgonia': True, 'MKL': True,
    'CUDA.jl': False, 'Candle': False, 'torch (R)': False, 'ND4J': False, 'Octave (en lugar de MATLAB)': False,
    'TorchSharp': False, 'Swift (sin Metal)': False, 'Breeze': False, 'KotlinDL': False,
}
if TODOS: ACTIVOS = {k: True for k in ACTIVOS}
# Lo pesado (Node, el motor de Ñ, Julia, Go, .NET, compilados…) va a /kaggle/temp, que NO se guarda
# en el Output; en /kaggle/working (el Output) solo quedan los CSV y el JSON de resultados, unos KB.
TRABAJO = '/kaggle/temp/comparar'
SALIDA = '/kaggle/working'

In [ ]:
# 2 · Node 22 y el proyecto (dataset subido, o git clone)
import os, subprocess, shutil, re, glob, time, json
os.makedirs(TRABAJO, exist_ok=True); os.makedirs(SALIDA, exist_ok=True)
# lo que versiones viejas de este cuaderno dejaban en el Output (el motor, los lenguajes): se borra
VIEJO = ['n', 'candle', 'candle-target', 'cs', 'dotnet', 'dotnet-install.sh', 'eigen', 'go', 'gocache', 'gopath', 'gorgonia',
         'julia-1.13.2', 'julia-depot', 'kotlinc', 'kotlinc.zip', 'kotlindl.jar', 'mkl', 'mklenv', 'mklpref', 'nuget', 'openblas',
         'scala-cli', 'swiftbench', 'tfjs', 'torchsharp'] + [os.path.basename(p) for p in glob.glob(f'{SALIDA}/node-v*-linux-x64')]
for v in VIEJO:
    r = f'{SALIDA}/{v}'
    if os.path.isdir(r) and not os.path.islink(r): shutil.rmtree(r, ignore_errors=True); print('borrado del Output:', r)
    elif os.path.exists(r): os.remove(r); print('borrado del Output:', r)
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C {TRABAJO}', shell=True, check=True)
    os.environ['PATH'] = f'{TRABAJO}/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())
PROYECTO = f'{TRABAJO}/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if os.path.exists(PROYECTO): print('ya estaba en', PROYECTO)
elif candidatos:
    shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules')); print('copiado desde', candidatos[0])
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/diegovelandiabarajas1-lang/Espanol-Like', PROYECTO], check=True)
os.chdir(PROYECTO)
COMP = f'{PROYECTO}/c1en/comparar'
if not os.path.exists(f'{COMP}/n.esl'):
    commit = subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
    raise SystemExit(f'falta c1en/comparar/ (los programas de la comparación). El repositorio está en «{commit}»: haz git push desde tu PC.')
print(subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
# el cuaderno y los programas tienen que ser de la misma versión: con PAPEL 'gpu' o 'cpu' los programas
# que miden CPU y GPU deben entender SOLO, si no el de GPU vuelve a medir la CPU (con media máquina)
if PAPEL != 'todo':
    viejos = [f for f in ['pytorch.py', 'cudajl.jl', 'rtorch.R', 'candle/src/main.rs', 'torchsharp/Program.cs']
              if 'SOLO' not in open(f'{COMP}/{f}', encoding='utf-8').read()]
    if viejos:
        commit = subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
        raise SystemExit(f'el proyecto está en «{commit}» y {", ".join(viejos)} son de antes de PAPEL: haz git push desde tu PC y vuelve a correr')
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!lscpu | grep -E '^(Model name|CPU\(s\)|Thread|Core)'

In [ ]:
# 3 · Ayudantes: instalar (con candado para apt), correr un programa y quedarse con sus líneas RES
import threading, pandas as pd
from concurrent.futures import ThreadPoolExecutor
ARG = ' '.join(['chica'] if ESCALA == 'chica' else [])
filas, estado, tiempos = [], {}, {}
candado_filas, candado_apt, candado_print = threading.Lock(), threading.Lock(), threading.Lock()
def decir(*a):
    with candado_print: print(*a, flush=True)
def sh(cmd, timeout=3600, env=None, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd,
                       env=dict(os.environ, **(env or {})))
    if r.returncode:
        raise RuntimeError(f'«{cmd[:140]}» salió con {r.returncode}:\n' + '\n'.join((r.stdout + r.stderr).strip().splitlines()[-15:]))
    return r.stdout
def apt(*paquetes):
    with candado_apt:                                       # apt-get no admite dos a la vez
        sh('apt-get update -qq', timeout=900)
        sh('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends ' + ' '.join(paquetes), timeout=1800)
HAY_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
N_GPU = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines()) if HAY_GPU else 0
assert PAPEL in ('gpu', 'cpu', 'todo'), "PAPEL es 'gpu', 'cpu' o 'todo'"
if PAPEL == 'gpu' and not HAY_GPU:
    raise SystemExit("PAPEL = 'gpu' pero esta sesión no tiene GPU: Settings → Accelerator → GPU T4 x2 (o cambia a PAPEL = 'cpu')")
if PAPEL == 'cpu' and HAY_GPU:
    print("(PAPEL = 'cpu' en una sesión con GPU: funciona, pero gasta horas de GPU sin usarla; mejor Accelerator → None)")
QUIERE_CPU = PAPEL in ('cpu', 'todo')
QUIERE_GPU = PAPEL in ('gpu', 'todo') and HAY_GPU
SOLO = {'cpu': 'cpu', 'gpu': 'gpu', 'todo': ''}[PAPEL]       # se les pasa a los programas que miden las dos cosas

# Los núcleos físicos (los hilos hermanos juntos), para repartirlos entre los dos trabajadores
nucleos = {}
for c in sorted(glob.glob('/sys/devices/system/cpu/cpu[0-9]*')):
    try:
        cid = (open(f'{c}/topology/physical_package_id').read().strip(), open(f'{c}/topology/core_id').read().strip())
        nucleos.setdefault(cid, []).append(int(c.rsplit('cpu', 1)[1]))
    except OSError: pass
grupos = list(nucleos.values()) or [[i] for i in range(os.cpu_count())]
if PAPEL == 'cpu':
    TRABAJADORES = [dict(id=1, gpu=None, cpus=sorted(sum(grupos, [])))]
elif (PAPEL == 'gpu' or MODO == 'paralelo') and N_GPU >= 2 and len(grupos) >= 2:
    mitad = len(grupos) // 2
    TRABAJADORES = [dict(id=1, gpu=0, cpus=sorted(sum(grupos[:mitad], []))), dict(id=2, gpu=1, cpus=sorted(sum(grupos[mitad:], [])))]
else:
    if PAPEL == 'gpu' or MODO == 'paralelo': print('(sin dos GPU o sin núcleos que repartir: uno detrás de otro)')
    TRABAJADORES = [dict(id=1, gpu=0, cpus=sorted(sum(grupos, [])))]
for t in TRABAJADORES:
    h = len(t['cpus'])
    gpu_env = {'CUDA_VISIBLE_DEVICES': '' if t['gpu'] is None else str(t['gpu']), 'C1EN_GPU': '0', 'SOLO': SOLO}
    t['env'] = dict(gpu_env, HILOS=str(h), OMP_NUM_THREADS=str(h), MKL_NUM_THREADS=str(h), OPENBLAS_NUM_THREADS=str(h),
                    GOMAXPROCS=str(h), RAYON_NUM_THREADS=str(h), JULIA_NUM_THREADS=str(h)) if len(TRABAJADORES) > 1 else gpu_env
    t['prefijo'] = f"taskset -c {','.join(map(str, t['cpus']))} " if len(TRABAJADORES) > 1 and shutil.which('taskset') else ''
    print(f"trabajador {t['id']}: GPU {t['gpu']} · CPU {t['cpus']}")
print('papel', PAPEL, '· GPU:', N_GPU, '· escala', ESCALA, '· ', 'dos a la vez' if len(TRABAJADORES) > 1 else 'uno detrás de otro')

## Los lenguajes: cómo se instala cada uno y qué se mide

Cada lenguaje tiene su `instalar()` (que deja lo que hace falta y devuelve rutas y variables) y sus tareas de medida (la orden que se corre, en qué carpeta y si usa la GPU). Nada se corre en esta celda: solo se describe.

In [ ]:
L = {}
def lenguaje(nombre, lengua, biblioteca, estimado_s, gpu=False):
    def reg(f):
        L[nombre] = dict(nombre=nombre, lenguaje=lengua, biblioteca=biblioteca, estimado=estimado_s, fn=f, gpu=gpu)
        return f
    return reg
# instalar(): devuelve una lista de tareas (etiqueta, orden, env, cwd, usa_gpu)

@lenguaje('Ñ', 'Ñ', 'C1EÑ', 90, gpu=True)
def _():
    sh('npm install koffi --no-audit --no-fund --loglevel=error', timeout=900, cwd=PROYECTO)
    sh('node motor-exp/util/construir.js', cwd=PROYECTO)
    if QUIERE_GPU:   # antes de medir: el DGEMM, la FFT y lo demás de NEÑ dan en la T4 los mismos bits que en jit
        r = subprocess.run('node motor-exp/pruebas/exp-nen-gpu.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.2) y la precisión simple y media: el SGEMM y paralelo sobre real32 dan en la T4 los mismos bits que en jit
        r = subprocess.run('node motor-exp/pruebas/exp-precision.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ real32/real16 en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.3) y las colas: dos colas con dependencias, errores al esperar, promesas, en la T4
        r = subprocess.run('node motor-exp/pruebas/exp-asincrono.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ colas en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.4) y paraleloBloques: compartida y barreras de verdad en la T4
        r = subprocess.run('node motor-exp/pruebas/exp-bloques.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ bloques en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.5) los enteros de máquina (u8…u32) y los bits en los núcleos, en la T4
        r = subprocess.run('node motor-exp/pruebas/exp-tipos.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ tipos de máquina en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.7) los atómicos de paralelo en la T4 (hilos(n, f) corre en la CPU)
        r = subprocess.run('node motor-exp/pruebas/exp-hilos.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ hilos y atómicos en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.8) las dos T4 desde un solo programa: paraleloRepartido, colaGpu(1), enGpu y copias entre ellas
        r = subprocess.run('node motor-exp/pruebas/exp-varias-gpu.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ varias GPU (las T4 que haya): ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.9) las plantillas: un núcleo genérico con una rama por tipo, en la T4
        r = subprocess.run('node motor-exp/pruebas/exp-plantillas.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ plantillas en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
        # (motor-exp N4.11) sinComprobar: sin comprobaciones en la T4, los mismos bits
        r = subprocess.run('node motor-exp/pruebas/exp-sin-comprobar.js --gpu', shell=True, capture_output=True, text=True, cwd=PROYECTO, timeout=1800)
        lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
        decir('   Ñ sinComprobar en la T4: ' + (lineas[-1].strip() if lineas else 'sin salida') + ''.join('\n      ' + l for l in lineas if l.lstrip().startswith('✗')))
    return [(f'Ñ · {d}', f'node motor-exp/util/n-node.js c1en/comparar/n.esl {d} {ARG}', {}, PROYECTO, d == 'gpu')
            for d in (['jit', 'nativo'] if QUIERE_CPU else []) + (['gpu'] if QUIERE_GPU else [])]

@lenguaje('PyTorch', 'Python', 'PyTorch', 60, gpu=True)
def _():
    sh('python -c "import torch"')
    return [('Python · PyTorch', f'python {COMP}/pytorch.py {ARG}', {}, PROYECTO, True)]

@lenguaje('Eigen', 'C++', 'Eigen', 20)
def _():
    apt('libeigen3-dev', 'g++')
    sh(f'g++ -O3 -march=native -fopenmp -I/usr/include/eigen3 {COMP}/eigen.cpp -o {TRABAJO}/eigen')
    return [('C++ · Eigen', f'{TRABAJO}/eigen {ARG}', {}, TRABAJO, False)]

@lenguaje('OpenBLAS', 'C', 'OpenBLAS', 15)
def _():
    apt('libopenblas-dev', 'gcc')
    sh(f'gcc -O3 -march=native -fopenmp {COMP}/openblas.c -o {TRABAJO}/openblas -lopenblas -lm')
    return [('C · OpenBLAS', f'{TRABAJO}/openblas {ARG}', {}, TRABAJO, False)]

@lenguaje('MKL', 'Fortran', 'Intel MKL', 15)
def _():
    apt('gfortran')
    # MKL de pip en una carpeta aparte (--prefix): el Python de Kaggle no trae ensurepip, así que sin venv.
    # Según el sistema, pip deja lib/ e include/ en M o en M/local: se buscan.
    M = f'{TRABAJO}/mklpref'
    if not glob.glob(f'{M}/**/libmkl_rt.so.*', recursive=True):
        sh(f'python -m pip install -q --prefix {M} --no-warn-script-location --break-system-packages mkl mkl-devel', timeout=1800)
    rt = sorted(glob.glob(f'{M}/**/libmkl_rt.so.*', recursive=True))
    dfti = glob.glob(f'{M}/**/mkl_dfti.f90', recursive=True)
    if not rt or not dfti: raise RuntimeError('no aparecieron libmkl_rt.so.* o mkl_dfti.f90 en la carpeta de MKL')
    LIB, INC = os.path.dirname(rt[-1]), os.path.dirname(dfti[0])
    D = f'{TRABAJO}/mkl'; os.makedirs(D, exist_ok=True)
    sh(f'gfortran -O3 -march=native -fopenmp -I{INC} {dfti[0]} {COMP}/mkl.f90 '
       f'-L{LIB} -l:{os.path.basename(rt[-1])} -Wl,-rpath,{LIB} -o {D}/mklf', cwd=D)
    return [('Fortran · MKL', f'{D}/mklf {ARG}', {}, D, False)]

@lenguaje('TensorFlow.js', 'JavaScript', 'TensorFlow.js', 45)
def _():
    D = f'{TRABAJO}/tfjs'; os.makedirs(D, exist_ok=True)
    sh('npm init -y >/dev/null && npm install @tensorflow/tfjs-node@4.22.0 --no-audit --no-fund --loglevel=error', cwd=D, timeout=1800)
    shutil.copy(f'{COMP}/tfjs.js', D)
    return [('JavaScript · TensorFlow.js', f'node tfjs.js {ARG}', {}, D, False)]

@lenguaje('Gorgonia', 'Go', 'Gorgonia', 60)
def _():
    if not os.path.exists(f'{TRABAJO}/go/bin/go'):
        ver = json.loads(sh('curl -fsSL "https://go.dev/dl/?mode=json"'))[0]['version']
        sh(f'curl -fsSL https://go.dev/dl/{ver}.linux-amd64.tar.gz | tar -xz -C {TRABAJO}', timeout=900)
    env = {'PATH': f'{TRABAJO}/go/bin:' + os.environ['PATH'], 'GOPATH': f'{TRABAJO}/gopath', 'GOCACHE': f'{TRABAJO}/gocache', 'GOFLAGS': '-mod=mod'}
    D = f'{TRABAJO}/gorgonia'; os.makedirs(D, exist_ok=True)
    shutil.copy(f'{COMP}/gorgonia.go', f'{D}/main.go')
    if not os.path.exists(f'{D}/go.mod'): sh('go mod init comparar', env=env, cwd=D)
    sh('go get gorgonia.org/tensor@v0.9.24 go4.org/unsafe/assume-no-moving-gc@latest && go build -o gorgonia .', env=env, cwd=D, timeout=1800)
    # una dependencia de Gorgonia (assume-no-moving-gc) se niega a arrancar con un Go más nuevo que ella
    # aunque Go no mueva la memoria: su propia variable dice «sé lo que hago» para esta versión de Go
    gover = re.match(r'go\d+\.\d+', sh('go env GOVERSION', env=env).strip()).group(0)
    return [('Go · Gorgonia', f'./gorgonia {ARG}', {'ASSUME_NO_MOVING_GC_UNSAFE_RISK_IT_WITH': gover}, D, False)]

@lenguaje('CUDA.jl', 'Julia', 'CUDA.jl', 120, gpu=True)
def _():
    J = f'{TRABAJO}/julia-1.13.2/bin/julia'
    if not os.path.exists(J):
        sh(f'curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-1.13.2-linux-x86_64.tar.gz | tar -xz -C {TRABAJO}', timeout=1800)
    env = {'JULIA_DEPOT_PATH': f'{TRABAJO}/julia-depot'}
    paquetes = '["CUDA", "FFTW"]; using CUDA, FFTW' if QUIERE_GPU else '["FFTW"]; using FFTW'      # en la de CPU, sin CUDA.jl
    sh(f'{J} -e \'using Pkg; Pkg.add({paquetes})\'', env=env, timeout=3600)
    if not QUIERE_GPU: env['SIN_GPU'] = '1'
    return [('Julia · CUDA.jl', f'{J} --threads=auto {COMP}/cudajl.jl {ARG}', env, TRABAJO, True)]

@lenguaje('Candle', 'Rust', 'Candle', 60, gpu=True)
def _():
    cargo = os.path.expanduser('~/.cargo/bin/cargo')
    if not os.path.exists(cargo): sh('curl -fsSL https://sh.rustup.rs | sh -s -- -y --profile minimal', timeout=1800)
    nvcc = shutil.which('nvcc') or ('/usr/local/cuda/bin/nvcc' if os.path.exists('/usr/local/cuda/bin/nvcc') else None)
    env = {'PATH': os.path.dirname(cargo) + ':/usr/local/cuda/bin:' + os.environ['PATH'], 'CUDA_COMPUTE_CAP': '75',
           'CARGO_TARGET_DIR': f'{TRABAJO}/candle-target'}
    D = f'{TRABAJO}/candle'
    if os.path.exists(D): shutil.rmtree(D)
    shutil.copytree(f'{COMP}/candle', D)
    rasgos = '--features cuda' if (QUIERE_GPU and nvcc) else ''
    if QUIERE_GPU and not nvcc: decir('   Candle: sin nvcc, solo CPU')
    sh(f'cargo build --release {rasgos}', env=env, cwd=D, timeout=3600)
    return [('Rust · Candle', f'{TRABAJO}/candle-target/release/comparar-candle {ARG}', {}, D, bool(rasgos))]

@lenguaje('torch (R)', 'R', 'torch', 90, gpu=True)
def _():
    if not shutil.which('Rscript'):
        apt('software-properties-common', 'dirmngr', 'wget', 'gnupg')
        with candado_apt:
            sh('wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc > /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc')
            sh('add-apt-repository -y "deb https://cloud.r-project.org/bin/linux/ubuntu $(lsb_release -cs)-cran40/"', timeout=900)
        apt('r-base-core', 'r-base-dev')
    tipo = 'cu128' if QUIERE_GPU else 'cpu'
    instalar = ('options(timeout = 1800); '
                'ver <- available.packages(repos = "https://cloud.r-project.org")["torch", "Version"]; '
                f'options(repos = c(torch = sprintf("https://torch-cdn.mlverse.org/packages/{tipo}/%s/", ver), CRAN = "https://cloud.r-project.org")); '
                'install.packages("torch"); library(torch); if (!torch_is_installed()) install_torch()')
    sh(f"Rscript -e '{instalar}'", env={'TORCH_INSTALL': '1'}, timeout=3600)
    return [('R · torch', f'Rscript {COMP}/rtorch.R {ARG}', {}, TRABAJO, True)]

@lenguaje('TorchSharp', 'C#', 'TorchSharp', 90, gpu=True)
def _():
    DN = f'{TRABAJO}/dotnet'
    if not os.path.exists(f'{DN}/dotnet'):
        sh(f'curl -fsSL https://dot.net/v1/dotnet-install.sh -o {TRABAJO}/dotnet-install.sh && bash {TRABAJO}/dotnet-install.sh --channel 8.0 --install-dir {DN}', timeout=1800)
    env = {'DOTNET_ROOT': DN, 'PATH': DN + ':' + os.environ['PATH'], 'DOTNET_CLI_TELEMETRY_OPTOUT': '1', 'DOTNET_NOLOGO': '1',
           'NUGET_PACKAGES': f'{TRABAJO}/nuget'}
    D = f'{TRABAJO}/torchsharp'
    if os.path.exists(D): shutil.rmtree(D)
    shutil.copytree(f'{COMP}/torchsharp', D)
    sh(f"dotnet add package {'TorchSharp-cuda-linux' if QUIERE_GPU else 'TorchSharp-cpu'}", env=env, cwd=D, timeout=3600)
    sh('dotnet build -c Release -o salida', env=env, cwd=D, timeout=1800)
    return [('C# · TorchSharp', f'dotnet salida/comparar.dll {ARG}', env, D, True)]

def jvm():
    if not shutil.which('javac'): apt('openjdk-17-jdk-headless')
    cs = f'{TRABAJO}/cs'
    with candado_apt:                                        # que dos JVM no bajen Coursier a la vez
        if not os.path.exists(cs):
            sh(f'curl -fsSL https://github.com/coursier/launchers/raw/master/cs-x86_64-pc-linux.gz | gzip -d > {cs}.tmp && chmod +x {cs}.tmp && mv {cs}.tmp {cs}', timeout=900)
    return cs
def jhilos(): return '-XX:ActiveProcessorCount=$HILOS ' if len(TRABAJADORES) > 1 else ''

@lenguaje('ND4J', 'Java', 'ND4J', 60)
def _():
    cs = jvm()
    cp = sh(f'{cs} fetch --classpath org.nd4j:nd4j-native:1.0.0-M2.1 org.nd4j:nd4j-native:1.0.0-M2.1,classifier=linux-x86_64', timeout=1800).strip()
    return [('Java · ND4J', f'java -Xmx12g {jhilos()}-cp "{cp}" {COMP}/Nd4jBench.java {ARG}', {}, TRABAJO, False)]

@lenguaje('Breeze', 'Scala', 'Breeze', 60)
def _():
    jvm(); apt('libopenblas-dev')                  # la libblas.so.3 que encuentra netlib
    sc = f'{TRABAJO}/scala-cli'
    if not os.path.exists(sc):
        sh(f'curl -fsSL https://github.com/VirtusLab/scala-cli/releases/latest/download/scala-cli-x86_64-pc-linux.gz | gzip -d > {sc} && chmod +x {sc}', timeout=900)
    sh(f'{sc} compile --server=false {COMP}/breeze.sc', timeout=1800)           # baja Scala y Breeze aquí, no al medir
    return [('Scala · Breeze', f'{sc} run --server=false {COMP}/breeze.sc -- {ARG}', {}, TRABAJO, False)]

@lenguaje('KotlinDL', 'Kotlin', 'KotlinDL', 60)
def _():
    cs = jvm()
    K = f'{TRABAJO}/kotlinc/bin/kotlinc'
    if not os.path.exists(K):
        rel = json.loads(sh('curl -fsSL https://api.github.com/repos/JetBrains/kotlin/releases/latest'))
        url = next(a['browser_download_url'] for a in rel['assets'] if re.match(r'kotlin-compiler-[\d.]+\.zip$', a['name']))
        sh(f'curl -fsSL -o {TRABAJO}/kotlinc.zip {url} && cd {TRABAJO} && python -m zipfile -e kotlinc.zip . && chmod +x kotlinc/bin/*', timeout=1800)
    cp = sh(f'{cs} fetch --classpath org.jetbrains.kotlinx:kotlin-deeplearning-tensorflow:0.5.2', timeout=1800).strip()
    sh(f'{K} {COMP}/kotlindl.kt -cp "{cp}" -include-runtime -d {TRABAJO}/kotlindl.jar', timeout=1800)
    return [('Kotlin · KotlinDL', f'java -Xmx12g {jhilos()}-cp "{TRABAJO}/kotlindl.jar:{cp}" KotlindlKt {ARG}', {}, TRABAJO, False)]

@lenguaje('Octave (en lugar de MATLAB)', 'MATLAB (Octave)', 'Octave', 40)
def _():
    apt('octave', 'libopenblas0')
    return [('MATLAB → Octave', f'octave-cli --no-gui {COMP}/octave.m {ARG}', {}, TRABAJO, False)]

@lenguaje('Swift (sin Metal)', 'Swift', 'Swift puro, sin Metal', 150)
def _():
    S = os.path.expanduser('~/.local/share/swiftly')
    if not shutil.which('swiftc') and not os.path.exists(f'{S}/bin/swiftc'):
        apt('binutils', 'git', 'gnupg2', 'libc6-dev', 'libcurl4-openssl-dev', 'libedit2', 'libpython3-dev', 'libsqlite3-0',
            'libxml2-dev', 'libz3-dev', 'pkg-config', 'tzdata', 'unzip', 'zlib1g-dev', 'libstdc++-12-dev', 'libgcc-12-dev')
        sh(f'cd {TRABAJO} && curl -fsSLO https://download.swift.org/swiftly/linux/swiftly-$(uname -m).tar.gz && tar zxf swiftly-$(uname -m).tar.gz '
           f'&& ./swiftly init --assume-yes --quiet-shell-followup', timeout=3600)
    env = {'PATH': f'{S}/bin:' + os.environ['PATH']}
    sh(f'swiftc -Ounchecked {COMP}/swift.swift -o {TRABAJO}/swiftbench', env=env, timeout=1800)
    return [('Swift (sin Metal)', f'{TRABAJO}/swiftbench {ARG}', {}, TRABAJO, False)]

# en el cuaderno de GPU solo entran los que tienen GPU; los demás se miden en el de CPU
def toca(k): return ACTIVOS.get(k) and (QUIERE_CPU or L[k]['gpu'])
print(len(L), 'lenguajes descritos · en este cuaderno (papel ' + PAPEL + '):', ', '.join(k for k in L if toca(k)))

## Fase 1 · Instalar y compilar, varios a la vez

Lo que más tarda es bajar y compilar. Se hace en paralelo (`INSTALAR_A_LA_VEZ`). Cada lenguaje dice al terminar si quedó listo o por qué no.

In [ ]:
TAREAS = []          # (estimado, etiqueta, lenguaje, biblioteca, orden, env, cwd, usa_gpu)
def instalar(nombre):
    d = L[nombre]; t0 = time.time()
    try:
        tareas = d['fn']()
        tiempos[nombre] = {'instalar_s': round(time.time() - t0)}
        with candado_filas:
            for (et, cmd, env, cwd, gpu) in tareas:
                if gpu and not HAY_GPU: gpu = False
                TAREAS.append((d['estimado'] / len(tareas), et, d['lenguaje'], d['biblioteca'], cmd, env, cwd, gpu, nombre))
        decir(f'✓ {nombre}: listo en {time.time() - t0:.0f} s')
    except Exception as e:
        estado[nombre] = '✗ al instalar: ' + str(e).splitlines()[0][:200]
        decir(f'✗ {nombre}: no se pudo instalar ({time.time() - t0:.0f} s)\n{e}')
for k in L:
    if not ACTIVOS.get(k): estado[k] = 'apagado'
    elif not toca(k): estado[k] = 'en el cuaderno de CPU'
t0 = time.time()
with ThreadPoolExecutor(max_workers=INSTALAR_A_LA_VEZ) as ex:
    list(ex.map(instalar, [k for k in L if toca(k)]))
print(f'\nfase 1: {time.time() - t0:.0f} s de pared · {len(TAREAS)} tareas de medida listas')

## Fase 2 · Medir, uno por GPU

Dos trabajadores (uno por T4, cada uno con su mitad de la CPU) sacan tareas de la cola, de la más larga a la más corta. La salida de cada tarea se escribe entera cuando termina, con el trabajador delante.

In [ ]:
import queue
cola = queue.Queue()
for t in sorted(TAREAS, key=lambda t: -t[0]): cola.put(t)
def correr_tarea(trab, tarea):
    _, et, lengua, bib, cmd, env, cwd, gpu, nombre = tarea
    t0 = time.time()
    salida = sh(trab['prefijo'] + cmd, timeout=5400, env=dict(env, **trab['env']), cwd=cwd)
    nuevas = []
    for l in salida.splitlines():
        if not l.startswith('RES,'): continue
        _, prueba, tam, disp, ms, huella, prec = l.strip().split(',')
        nuevas.append(dict(entrada=et, lenguaje=lengua, biblioteca=bib, prueba=prueba, tam=tam, dispositivo=disp, ms=float(ms),
                           huella=float(huella), precision=prec, trabajador=trab['id'], gpu_fisica=trab['gpu'] if disp == 'gpu' else None,
                           hilos=len(trab['cpus']), modo='paralelo' if len(TRABAJADORES) > 1 else 'secuencial', papel=PAPEL))
    with candado_filas:
        filas.extend(nuevas)
        tiempos.setdefault(nombre, {})[et] = round(time.time() - t0)
    comentarios = [l for l in salida.splitlines() if l.startswith('#')]
    decir(f"[T{trab['id']}·GPU{trab['gpu']}] ✓ {et}: {len(nuevas)} medidas en {time.time() - t0:.0f} s" + ''.join('\n      ' + c for c in comentarios))
    return len(nuevas)
def trabajador(trab):
    while True:
        try: tarea = cola.get_nowait()
        except queue.Empty: return
        nombre = tarea[8]
        try:
            n = correr_tarea(trab, tarea)
            with candado_filas:
                estado[nombre] = estado.get(nombre, '') if estado.get(nombre, '').startswith('✗') else '✓'
        except Exception as e:
            with candado_filas: estado[nombre] = f'✗ al medir {tarea[1]}: ' + str(e).splitlines()[0][:200]
            decir(f"[T{trab['id']}·GPU{trab['gpu']}] ✗ {tarea[1]}:\n{e}")
t0 = time.time()
hilos_ = [threading.Thread(target=trabajador, args=(t,)) for t in TRABAJADORES]
for h in hilos_: h.start()
for h in hilos_: h.join()
print(f'\nfase 2: {time.time() - t0:.0f} s de pared con {len(TRABAJADORES)} trabajador(es) · {len(filas)} medidas')
suma_tareas = sum(v for d in tiempos.values() for k, v in d.items() if k != 'instalar_s')
print(f'(una detrás de otra habrían sido ~{suma_tareas} s)')

## Juntar con el otro cuaderno

Si añadiste como entrada la salida del otro cuaderno (*Add Input → Your Work*), aquí se junta su CSV con este. Si no, la tabla es solo de esta máquina. Si la sesión se reinició al añadir la entrada, no hace falta medir otra vez: corre las tres primeras celdas de código (parámetros, proyecto y ayudas) y luego de aquí hacia abajo; con las salidas de **los dos** cuadernos como entrada, junta las dos.

In [ ]:
import glob
filas = globals().get('filas', [])      # sin la fase 2 (sesión nueva): solo junta lo que haya en las entradas
df_aqui = pd.DataFrame(filas)
os.makedirs(SALIDA, exist_ok=True)
if filas: df_aqui.to_csv(f'{SALIDA}/comparar-lenguajes-{PAPEL}.csv', index=False)
otros, vistos = [], {PAPEL} if filas else set()
for f in sorted(glob.glob('/kaggle/input/**/comparar-lenguajes-*.csv', recursive=True)):
    if 'resumen' in f or 'junto' in f: continue
    o = pd.read_csv(f, dtype={'tam': str})
    if 'papel' in o and set(o['papel']) & vistos: continue
    vistos |= set(o.get('papel', []))
    otros.append(o); print('juntado:', f, len(o), 'medidas', sorted(set(o.get('papel', ['?']))))
filas_todas = pd.concat([df_aqui] + otros, ignore_index=True).to_dict('records') if otros else filas
if not filas_todas: raise SystemExit('no hay medidas: corre la fase 2 o añade como entrada la salida de los cuadernos')
print(f'{len(filas_todas)} medidas en total ({len(filas)} de esta máquina)')

## ¿Calculan lo mismo? La huella contra NumPy

Cada huella se compara con la de NumPy en doble precisión (`c1en/comparar/referencia.py`). En doble precisión la diferencia relativa tiene que ser pequeñísima (≤ 1e-9; las sumas de 16 millones de números, en distinto orden, se quedan en ~1e-12); en float32 (TensorFlow.js, la FFT de KotlinDL), ~1e-4.

In [ ]:
import importlib.util, numpy as np
spec = importlib.util.spec_from_file_location('referencia', f'{COMP}/referencia.py'); ref = importlib.util.module_from_spec(spec); spec.loader.exec_module(ref)
REF = ref.huellas((128, 256), 65536, 256, 1024, 16384) if ESCALA == 'chica' else ref.huellas()
df = pd.DataFrame(filas_todas)
df['tam'] = df['tam'].astype(str)
df['referencia'] = [REF.get((p, t), np.nan) for p, t in zip(df['prueba'], df['tam'])]
df['dif_rel'] = (df['huella'] - df['referencia']).abs() / df['referencia'].abs()
df['bien'] = np.where(df['precision'] == 'f32', df['dif_rel'] < 1e-3, df['dif_rel'] < 1e-9)
malos = df[~df['bien']]
print(f'{len(df)} medidas · {int(df["bien"].sum())} con la huella de NumPy · {len(malos)} distintas')
if len(malos): print(malos[['entrada', 'prueba', 'tam', 'dispositivo', 'huella', 'referencia', 'dif_rel']].to_string(index=False))
print('\nEstado y tiempos de cada lenguaje (de esta sesión):')
for k in globals().get('L', {}):
    print(f'  {k:30s} {estado.get(k, "?"):12s} {tiempos.get(k, "")}')

## La tabla

Milisegundos, la mejor de 3 (menos es mejor). Cada columna es *lenguaje · biblioteca (dispositivo)*. Con los dos cuadernos, las de CPU salen de la sesión de CPU (CPU entera) y las de GPU de la de T4 x2 (una T4 entera cada una); con `PAPEL = 'todo'` y modo paralelo, las de CPU son con media CPU (lo dice la cabecera). Debajo, para cada prueba, quién va primero, cuántas veces más lento va cada uno y el puesto de Ñ.

In [ ]:
modo = ', '.join(sorted(set(f"{p}:{m}" for p, m in zip(df.get('papel', PAPEL), df['modo'])))) if len(df) else MODO
hilos_cpu = sorted(int(h) for h in df[df['dispositivo'] != 'gpu']['hilos'].unique()) if len(df) else []
print(f'máquinas y modos: {modo} · hilos de CPU por lenguaje en las pruebas de CPU: {hilos_cpu}\n')
df['clase'] = np.where(df['dispositivo'] == 'gpu', 'GPU', 'CPU')
df['col'] = df['entrada'].str.replace('Ñ · ', 'Ñ ', regex=False) + df.apply(lambda r: '' if r['lenguaje'] == 'Ñ' else f" ({r['dispositivo']}{', f32' if r['precision'] == 'f32' else ''})", axis=1)
orden_tam = lambda s: s.map(lambda t: float(str(t).split('x')[0]))
ancho = df.pivot_table(index=['prueba', 'tam'], columns='col', values='ms', aggfunc='min')
ancho = ancho.reset_index().sort_values(['prueba', 'tam'], key=lambda s: orden_tam(s) if s.name == 'tam' else s).set_index(['prueba', 'tam'])
pd.set_option('display.width', 400); pd.set_option('display.max_columns', 60)
print(ancho.round(2).T.to_string())
print()
# los rankings, CPU con CPU y GPU con GPU (cada uno salió de una sola máquina)
for clase in ['CPU', 'GPU']:
    dc = df[df['clase'] == clase]
    if not len(dc): continue
    print(f'══════ {clase}')
    for (p, t), g in dc.groupby(['prueba', 'tam'], sort=False):
        g = g.sort_values('ms'); mejor = g['ms'].iloc[0]
        linea = ' · '.join(f"{r['col']} {r['ms']:.1f}" + ('' if k == 0 else f" (×{r['ms'] / mejor:.1f})") for k, (_, r) in enumerate(g.iterrows()))
        pos = [k + 1 for k, (_, r) in enumerate(g.iterrows()) if r['lenguaje'] == 'Ñ']
        print(f'{p} {t}: Ñ en el puesto {pos} de {len(g)}\n   {linea}\n')

In [ ]:
import matplotlib.pyplot as plt
pruebas = list(dict.fromkeys(zip(df['prueba'], df['tam'])))
fig, ejes = plt.subplots(len(pruebas), 1, figsize=(14, 3.2 * len(pruebas)))
ejes = np.atleast_1d(ejes)
for ax, (p, t) in zip(ejes, pruebas):
    g = df[(df['prueba'] == p) & (df['tam'] == t)].sort_values('ms')
    colores = ['#c0392b' if l == 'Ñ' else ('#2e86c1' if d == 'gpu' else '#7f8c8d') for l, d in zip(g['lenguaje'], g['dispositivo'])]
    ax.barh(g['col'], g['ms'], color=colores); ax.set_xscale('log'); ax.invert_yaxis()
    ax.set_title(f'{p} · {t}  (ms, escala log; rojo Ñ, azul GPU, gris CPU; modo {modo})', fontsize=10); ax.tick_params(axis='y', labelsize=8)
plt.tight_layout(); plt.show()

mm = df[df['prueba'] == 'matmul'].copy()
mm['GFLOP/s'] = 2 * mm['tam'].astype(float) ** 3 / (mm['ms'] / 1000) / 1e9
print(mm.pivot_table(index='col', columns='tam', values='GFLOP/s').sort_values(mm['tam'].max(), ascending=False).round(1).to_string())

In [ ]:
# Los CSV para descargar (Output) — y para mandármelos
# comparar-lenguajes-{papel}.csv (lo de esta máquina) ya se guardó al juntar; aquí, la tabla conjunta
df.drop(columns=['col']).to_csv(f'{SALIDA}/comparar-lenguajes-junto.csv', index=False)
ancho.to_csv(f'{SALIDA}/comparar-lenguajes-resumen-{PAPEL}.csv')
json.dump(dict(papel=PAPEL, escala=ESCALA, modo=modo, trabajadores=[{k: t[k] for k in ('id', 'gpu', 'cpus')} for t in TRABAJADORES], estado=estado, tiempos=tiempos,
               commit=subprocess.run(['git', 'log', '-1', '--format=%h'], capture_output=True, text=True).stdout.strip()),
          open(f'{SALIDA}/comparar-lenguajes-info-{PAPEL}.json', 'w'), ensure_ascii=False, indent=1)
for f in [f'comparar-lenguajes-{PAPEL}.csv', 'comparar-lenguajes-junto.csv', f'comparar-lenguajes-resumen-{PAPEL}.csv', f'comparar-lenguajes-info-{PAPEL}.json']:
    print(f'{SALIDA}/{f}: {os.path.getsize(f"{SALIDA}/{f}") / 1024:.1f} KB')
print('Output entero:', subprocess.run(['du', '-sh', SALIDA], capture_output=True, text=True).stdout.split()[0])

## Cómo leerlo

- **Lo justo de la comparación:** todos en doble precisión salvo TensorFlow.js (float32) y la FFT de KotlinDL; todos con los datos ya en su dispositivo y el resultado sin copiar. PyTorch, R torch y TorchSharp usan cuBLAS y cuFFT en la GPU (con FMA); C1EÑ no usa FMA para dar los mismos bits que la CPU, así que en `matmul` de GPU su techo es la mitad.
- **Dos cuadernos:** las pruebas de CPU salen todas de la sesión de CPU, con la CPU entera y una tras otra; las de GPU, de la sesión con las dos T4, una por GPU. Por eso los rankings van separados: CPU con CPU, GPU con GPU.
- **CPU:** Eigen, OpenBLAS, MKL y PyTorch usan AVX/FMA y todos sus núcleos; los de la JVM y Go dependen de su BLAS; Swift va en un hilo a propósito (es Swift puro).
- **Lo que no está:** OpenBLAS, Gorgonia, Candle, ND4J y Swift no traen FFT.
- Si un lenguaje sale ✗ en el estado, su error está en la fase 1 o 2: mándamelo y lo arreglo (C++, C, Fortran, Octave y Ñ se probaron antes de entregarlos, TensorFlow.js con su motor de JavaScript puro; los demás se prueban aquí por primera vez).